<a href="https://colab.research.google.com/github/Kobby24/intro_to_AI_py/blob/main/GSE7390_BRCA_xAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.isotonic import IsotonicRegression

## QuantileNormalizer

In [ ]:
class QuantileNormalizer:
    """
    Quantile Normalization for microarray data.

    Transforms each sample to have identical distribution by mapping
    to a reference distribution computed as the mean across all samples.

    Reference: Eq. (1) and (2) in paper
    """

    def __init__(self):
        self.reference_distribution = None

    def fit(self, X):
        """
        Compute reference distribution from training data.

        Args:
            X: Expression matrix (samples x genes)
        """
        X = np.array(X)
        n_samples, n_genes = X.shape

        # Sort each sample and compute mean across samples at each rank
        sorted_data = np.sort(X, axis=1)
        self.reference_distribution = np.mean(sorted_data, axis=0)

        return self

    def transform(self, X):
        """
        Apply quantile normalization.

        Args:
            X: Expression matrix (samples x genes)

        Returns:
            Normalized expression matrix
        """
        X = np.array(X)
        n_samples, n_genes = X.shape
        X_normalized = np.zeros_like(X)

        for i in range(n_samples):
            # Get ranks of original values
            ranks = stats.rankdata(X[i], method='ordinal') - 1
            # Map to reference distribution
            X_normalized[i] = self.reference_distribution[ranks.astype(int)]

        return X_normalized

    def fit_transform(self, X):
        """Fit and transform in one step."""
        self.fit(X)
        return self.transform(X)


## ComBatCorrection

In [ ]:
class ComBatCorrection:
    """
    ComBat batch effect correction using empirical Bayes estimation.

    Removes systematic batch effects while preserving biological variation.

    Reference: Eq. (3) and (4) in paper
    """

    def __init__(self):
        self.grand_mean = None
        self.batch_means = None
        self.batch_vars = None
        self.gamma_star = None
        self.delta_star = None

    def fit(self, X, batch_labels):
        """
        Estimate batch effect parameters.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignment for each sample
        """
        X = np.array(X)
        batch_labels = np.array(batch_labels)
        unique_batches = np.unique(batch_labels)

        n_samples, n_genes = X.shape

        # Compute grand mean
        self.grand_mean = np.mean(X, axis=0)

        # Compute batch-specific parameters
        self.batch_means = {}
        self.batch_vars = {}

        for batch in unique_batches:
            mask = batch_labels == batch
            batch_data = X[mask]
            self.batch_means[batch] = np.mean(batch_data, axis=0)
            self.batch_vars[batch] = np.var(batch_data, axis=0) + 1e-10

        # Empirical Bayes estimation of batch effects
        self.gamma_star = {}
        self.delta_star = {}

        for batch in unique_batches:
            # Additive batch effect (standardized)
            self.gamma_star[batch] = self.batch_means[batch] - self.grand_mean
            # Multiplicative batch effect
            pooled_var = np.mean([self.batch_vars[b] for b in unique_batches], axis=0)
            self.delta_star[batch] = np.sqrt(self.batch_vars[batch] / (pooled_var + 1e-10))

        return self

    def transform(self, X, batch_labels):
        """
        Apply batch correction.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignment for each sample

        Returns:
            Batch-corrected expression matrix
        """
        X = np.array(X)
        batch_labels = np.array(batch_labels)
        X_corrected = np.zeros_like(X)

        for i, batch in enumerate(batch_labels):
            if batch in self.gamma_star:
                # Apply correction: (Y - gamma*) / delta* + grand_mean
                X_corrected[i] = (X[i] - self.gamma_star[batch]) / (self.delta_star[batch] + 1e-10) + self.grand_mean
            else:
                X_corrected[i] = X[i]

        return X_corrected

    def fit_transform(self, X, batch_labels):
        """Fit and transform in one step."""
        self.fit(X, batch_labels)
        return self.transform(X, batch_labels)


## KNNImputer

In [ ]:
class KNNImputer:
    """
    K-Nearest Neighbors imputation with weighted averaging.

    Reference: Eq. (5) and (6) in paper
    """

    def __init__(self, n_neighbors=10, sigma=1.0):
        """
        Args:
            n_neighbors: Number of nearest neighbors (K)
            sigma: Bandwidth parameter for Gaussian weights
        """
        self.n_neighbors = n_neighbors
        self.sigma = sigma

    def fit_transform(self, X):
        """
        Impute missing values using KNN.

        Args:
            X: Expression matrix with possible NaN values

        Returns:
            Imputed expression matrix
        """
        X = np.array(X, dtype=float)
        n_samples, n_genes = X.shape

        # Find missing values
        missing_mask = np.isnan(X)

        if not np.any(missing_mask):
            return X

        X_imputed = X.copy()

        for i in range(n_samples):
            if not np.any(missing_mask[i]):
                continue

            # Find non-missing genes for this sample
            valid_genes = ~missing_mask[i]
            missing_genes = missing_mask[i]

            # Compute distances to other samples using valid genes
            distances = []
            for j in range(n_samples):
                if i == j or np.any(missing_mask[j, valid_genes]):
                    distances.append(np.inf)
                else:
                    dist = np.sqrt(np.sum((X[i, valid_genes] - X[j, valid_genes])**2))
                    distances.append(dist)

            distances = np.array(distances)

            # Get K nearest neighbors
            neighbor_indices = np.argsort(distances)[:self.n_neighbors]
            neighbor_distances = distances[neighbor_indices]

            # Compute Gaussian weights
            weights = np.exp(-neighbor_distances**2 / (2 * self.sigma**2))
            weights = weights / (np.sum(weights) + 1e-10)

            # Impute missing values
            for gene_idx in np.where(missing_genes)[0]:
                neighbor_values = X[neighbor_indices, gene_idx]
                valid_neighbor_mask = ~np.isnan(neighbor_values)
                if np.any(valid_neighbor_mask):
                    X_imputed[i, gene_idx] = np.sum(
                        weights[valid_neighbor_mask]
                        * neighbor_values[valid_neighbor_mask]
                    ) / np.sum(weights[valid_neighbor_mask])
                else:
                    X_imputed[i, gene_idx] = np.nanmean(X[:, gene_idx])
        return X_imputed


## AttentionFeatureSelector

In [ ]:
class AttentionFeatureSelector:
    """
    Attention-weighted feature selection mechanism.

    Computes attention scores that quantify discriminative importance of each gene
    based on statistical embeddings and mutual information.

    Reference: Eq. (7), (8), and (9) in paper
    """

    def __init__(self, embedding_dim=64, mad_threshold_percentile=25):
        """
        Args:
            embedding_dim: Dimension of gene embeddings (d)
            mad_threshold_percentile: Percentile for MAD filtering threshold
        """
        self.embedding_dim = embedding_dim
        self.mad_threshold_percentile = mad_threshold_percentile
        self.attention_scores = None
        self.selected_genes = None
        self.W_e = None
        self.b_e = None
        self.W_h = None
        self.b_h = None
        self.w = None

    def _compute_statistics(self, X, y):
        """
        Compute statistical features for each gene.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels

        Returns:
            Statistical features (genes x 5): [mean, std, skewness, kurtosis, MI]
        """
        n_genes = X.shape[1]
        stats_features = np.zeros((n_genes, 5))

        for j in range(n_genes):
            gene_expr = X[:, j]
            stats_features[j, 0] = np.mean(gene_expr)  # Mean
            stats_features[j, 1] = np.std(gene_expr)   # Std
            stats_features[j, 2] = stats.skew(gene_expr)  # Skewness
            stats_features[j, 3] = stats.kurtosis(gene_expr)  # Kurtosis

        # Compute mutual information
        mi_scores = mutual_info_classif(X, y, random_state=42)
        stats_features[:, 4] = mi_scores

        # Normalize features
        stats_features = (stats_features - np.mean(stats_features, axis=0)) / (np.std(stats_features, axis=0) + 1e-10)

        return stats_features

    def _compute_embeddings(self, stats_features):
        """
        Compute gene embeddings from statistical features.

        Reference: Eq. (8)

        Args:
            stats_features: Statistical features (genes x 5)

        Returns:
            Gene embeddings (genes x embedding_dim)
        """
        n_genes = stats_features.shape[0]

        # Initialize learnable parameters (using random initialization)
        np.random.seed(42)
        self.W_e = np.random.randn(5, self.embedding_dim) * 0.1
        self.b_e = np.zeros(self.embedding_dim)

        # Compute embeddings: h_j = sigmoid(W_e @ stats + b_e)
        embeddings = 1 / (1 + np.exp(-(stats_features @ self.W_e + self.b_e)))

        return embeddings

    def _compute_attention_scores(self, embeddings):
        """
        Compute attention scores for each gene.

        Reference: Eq. (7)

        Args:
            embeddings: Gene embeddings (genes x embedding_dim)

        Returns:
            Attention scores (genes,)
        """
        n_genes = embeddings.shape[0]

        # Initialize attention parameters
        np.random.seed(43)
        self.W_h = np.random.randn(self.embedding_dim, self.embedding_dim) * 0.1
        self.b_h = np.zeros(self.embedding_dim)
        self.w = np.random.randn(self.embedding_dim) * 0.1

        # Compute attention: a_j = softmax(w^T * tanh(W_h @ h_j + b_h))
        hidden = np.tanh(embeddings @ self.W_h + self.b_h)
        scores = hidden @ self.w

        # Softmax normalization
        exp_scores = np.exp(scores - np.max(scores))
        attention_scores = exp_scores / np.sum(exp_scores)

        return attention_scores

    def fit(self, X, y):
        """
        Compute attention scores and select top genes.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels
        """
        X = np.array(X)
        y = np.array(y)
        n_genes = X.shape[1]

        # MAD filtering
        mad_values = np.median(np.abs(X - np.median(X, axis=0)), axis=0)
        mad_threshold = np.percentile(mad_values, self.mad_threshold_percentile)
        self.high_mad_genes = mad_values > mad_threshold

        print(f"MAD filtering: {np.sum(self.high_mad_genes)} genes pass threshold")

        # Compute statistics for genes passing MAD filter
        X_filtered = X[:, self.high_mad_genes]
        stats_features = self._compute_statistics(X_filtered, y)

        # Compute embeddings
        embeddings = self._compute_embeddings(stats_features)

        # Compute attention scores
        attention_filtered = self._compute_attention_scores(embeddings)

        # Map back to full gene set
        self.attention_scores = np.zeros(n_genes)
        self.attention_scores[self.high_mad_genes] = attention_filtered

        return self

    def get_top_genes(self, n_genes):
        """
        Get indices of top n genes by attention score.

        Args:
            n_genes: Number of genes to select

        Returns:
            Indices of selected genes
        """
        return np.argsort(self.attention_scores)[-n_genes:]

    def transform(self, X, selected_indices):
        """
        Select genes based on provided indices.

        Args:
            X: Expression matrix
            selected_indices: Indices of genes to keep

        Returns:
            Reduced expression matrix
        """
        return X[:, selected_indices]


## AdaptiveEnsembleClassifier

In [ ]:
class AdaptiveEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Adaptive Ensemble Classifier combining GBM, RF, and SVM.

    Features:
    - Gradient Boosting Machine for nonlinear interactions
    - Random Forest for variance reduction
    - Support Vector Machine for high-dimensional effectiveness
    - Adaptive weight optimization using SLSQP
    - Probability calibration using isotonic regression

    Reference: Eq. (10)-(18) in paper
    """

    def __init__(self,
                 gbm_params=None,
                 rf_params=None,
                 svm_params=None,
                 calibrate=True,
                 random_state=42):
        """
        Args:
            gbm_params: Parameters for Gradient Boosting
            rf_params: Parameters for Random Forest
            svm_params: Parameters for SVM
            calibrate: Whether to calibrate probabilities
            random_state: Random seed
        """
        self.gbm_params = gbm_params or {
            'learning_rate': 0.1,
            'max_depth': 6,
            'n_estimators': 200,
            'random_state': random_state
        }
        self.rf_params = rf_params or {
            'n_estimators': 500,
            'max_features': 'sqrt',
            'random_state': random_state
        }
        self.svm_params = svm_params or {
            'C': 1.0,
            'kernel': 'rbf',
            'gamma': 'scale',
            'probability': True,
            'random_state': random_state
        }
        self.calibrate = calibrate
        self.random_state = random_state

        # Initialize classifiers
        self.gbm = GradientBoostingClassifier(**self.gbm_params)
        self.rf = RandomForestClassifier(**self.rf_params)
        self.svm = SVC(**self.svm_params)

        # Ensemble weights (optimized during training)
        self.weights = np.array([1/3, 1/3, 1/3])

        # Calibration models
        self.calibrators = {}

        # Training history
        self.training_history = {'gbm_loss': [], 'rf_oob': [], 'svm_loss': []}

    def fit(self, X, y, X_val=None, y_val=None):
        """
        Train ensemble classifiers and optimize weights.

        Args:
            X: Training features
            y: Training labels
            X_val: Validation features (optional, for weight optimization)
            y_val: Validation labels (optional)
        """
        X = np.array(X)
        y = np.array(y)

        print("Training Gradient Boosting...")
        self.gbm.fit(X, y)

        print("Training Random Forest...")
        self.rf.fit(X, y)

        print("Training SVM...")
        self.svm.fit(X, y)

        # Record training history for GBM
        if hasattr(self.gbm, 'train_score_'):
            self.training_history['gbm_loss'] = self.gbm.train_score_.tolist()

        # Record OOB score for RF
        if hasattr(self.rf, 'oob_score_'):
            self.training_history['rf_oob'] = [self.rf.oob_score_]

        # Optimize weights if validation data provided
        if X_val is not None and y_val is not None:
            self._optimize_weights(X_val, y_val)
        else:
            # Use cross-validation to optimize weights
            self._optimize_weights_cv(X, y)

        # Calibrate probabilities
        if self.calibrate:
            self._fit_calibrators(X, y)

        return self

    def _optimize_weights(self, X_val, y_val):
        """
        Optimize ensemble weights using SLSQP.

        Reference: Eq. (17)
        """
        def cross_entropy_loss(weights):
            """Compute cross-entropy loss for weight combination."""
            weights = weights / np.sum(weights)  # Normalize

            # Get predictions from each classifier
            p_gbm = self.gbm.predict_proba(X_val)
            p_rf = self.rf.predict_proba(X_val)
            p_svm = self.svm.predict_proba(X_val)

            # Weighted combination
            p_ensemble = weights[0] * p_gbm + weights[1] * p_rf + weights[2] * p_svm
            p_ensemble = np.clip(p_ensemble, 1e-10, 1 - 1e-10)

            # Cross-entropy loss
            n_samples = len(y_val)
            loss = 0
            for i in range(n_samples):
                loss -= np.log(p_ensemble[i, y_val[i]])

            return loss / n_samples

        # Constraints: weights sum to 1, all non-negative
        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        bounds = [(0, 1), (0, 1), (0, 1)]

        # Optimize
        result = minimize(
            cross_entropy_loss,
            x0=np.array([1/3, 1/3, 1/3]),
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        self.weights = result.x / np.sum(result.x)
        print(f"Optimized weights: GBM={self.weights[0]:.3f}, RF={self.weights[1]:.3f}, SVM={self.weights[2]:.3f}")

    def _optimize_weights_cv(self, X, y, cv=3):
        """Optimize weights using cross-validation."""
        kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=self.random_state)

        all_preds_gbm = []
        all_preds_rf = []
        all_preds_svm = []
        all_labels = []

        for train_idx, val_idx in kfold.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            # Train temporary classifiers
            gbm_temp = GradientBoostingClassifier(**self.gbm_params)
            rf_temp = RandomForestClassifier(**self.rf_params)
            svm_temp = SVC(**self.svm_params)

            gbm_temp.fit(X_train, y_train)
            rf_temp.fit(X_train, y_train)
            svm_temp.fit(X_train, y_train)

            all_preds_gbm.append(gbm_temp.predict_proba(X_val))
            all_preds_rf.append(rf_temp.predict_proba(X_val))
            all_preds_svm.append(svm_temp.predict_proba(X_val))
            all_labels.append(y_val)

        # Concatenate predictions
        p_gbm = np.vstack(all_preds_gbm)
        p_rf = np.vstack(all_preds_rf)
        p_svm = np.vstack(all_preds_svm)
        y_all = np.concatenate(all_labels)

        def cross_entropy_loss(weights):
            weights = weights / np.sum(weights)
            p_ensemble = weights[0] * p_gbm + weights[1] * p_rf + weights[2] * p_svm
            p_ensemble = np.clip(p_ensemble, 1e-10, 1 - 1e-10)
            loss = 0
            for i in range(len(y_all)):
                loss -= np.log(p_ensemble[i, y_all[i]])
            return loss / len(y_all)

        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        bounds = [(0, 1), (0, 1), (0, 1)]

        result = minimize(
            cross_entropy_loss,
            x0=np.array([1/3, 1/3, 1/3]),
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        self.weights = result.x / np.sum(result.x)
        print(f"Optimized weights (CV): GBM={self.weights[0]:.3f}, RF={self.weights[1]:.3f}, SVM={self.weights[2]:.3f}")

    def _fit_calibrators(self, X, y):
        """
        Fit isotonic regression calibrators.

        Reference: Eq. (18)
        """
        # Get uncalibrated probabilities
        p_ensemble = self._get_uncalibrated_proba(X)

        n_classes = p_ensemble.shape[1]

        for c in range(n_classes):
            y_binary = (y == c).astype(int)
            calibrator = IsotonicRegression(out_of_bounds='clip')
            calibrator.fit(p_ensemble[:, c], y_binary)
            self.calibrators[c] = calibrator

    def _get_uncalibrated_proba(self, X):
        """Get uncalibrated ensemble probabilities."""
        p_gbm = self.gbm.predict_proba(X)
        p_rf = self.rf.predict_proba(X)
        p_svm = self.svm.predict_proba(X)

        p_ensemble = self.weights[0] * p_gbm + self.weights[1] * p_rf + self.weights[2] * p_svm
        return p_ensemble

    def predict_proba(self, X):
        """
        Predict class probabilities.

        Args:
            X: Features

        Returns:
            Calibrated class probabilities
        """
        p_ensemble = self._get_uncalibrated_proba(X)

        if self.calibrate and self.calibrators:
            p_calibrated = np.zeros_like(p_ensemble)
            for c, calibrator in self.calibrators.items():
                p_calibrated[:, c] = calibrator.predict(p_ensemble[:, c])
            # Normalize
            p_calibrated = p_calibrated / (np.sum(p_calibrated, axis=1, keepdims=True) + 1e-10)
            return p_calibrated

        return p_ensemble

    def predict(self, X):
        """Predict class labels."""
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

    def get_feature_importance(self):
        """Get aggregated feature importance from ensemble."""
        # GBM importance
        gbm_imp = self.gbm.feature_importances_

        # RF importance
        rf_imp = self.rf.feature_importances_

        # Aggregate with ensemble weights
        importance = self.weights[0] * gbm_imp + self.weights[1] * rf_imp
        importance = importance / np.sum(importance)

        return importance


## MICROAI

In [ ]:
class MICROAI:
    """
    MICRO-AI: Complete framework for DNA microarray analysis and disease detection.

    Integrates:
    - Quantile normalization
    - ComBat batch effect correction
    - KNN missing value imputation
    - Attention-weighted feature selection
    - RFECV for optimal gene subset selection
    - Adaptive ensemble classification
    - Probability calibration
    """

    def __init__(self,
                 n_selected_features=150,
                 n_neighbors_impute=10,
                 embedding_dim=64,
                 mad_percentile=25,
                 rfecv_step=0.1,
                 cv_folds=5,
                 calibrate=True,
                 random_state=42):
        """
        Args:
            n_selected_features: Target number of selected genes
            n_neighbors_impute: K for KNN imputation
            embedding_dim: Dimension for attention embeddings
            mad_percentile: Percentile threshold for MAD filtering
            rfecv_step: Step size for RFECV
            cv_folds: Number of cross-validation folds
            calibrate: Whether to calibrate probabilities
            random_state: Random seed for reproducibility
        """
        self.n_selected_features = n_selected_features
        self.n_neighbors_impute = n_neighbors_impute
        self.embedding_dim = embedding_dim
        self.mad_percentile = mad_percentile
        self.rfecv_step = rfecv_step
        self.cv_folds = cv_folds
        self.calibrate = calibrate
        self.random_state = random_state

        # Components (initialized during fit)
        self.quantile_normalizer = QuantileNormalizer()
        self.combat_corrector = None
        self.knn_imputer = KNNImputer(n_neighbors=n_neighbors_impute)
        self.attention_selector = AttentionFeatureSelector(
            embedding_dim=embedding_dim,
            mad_threshold_percentile=mad_percentile
        )
        self.scaler = StandardScaler()
        self.ensemble = None

        # Selected gene indices
        self.selected_genes = None
        self.gene_names = None

        # Label encoder
        self.label_encoder = LabelEncoder()

        # Training info
        self.n_classes = None
        self.training_info = {}

    def preprocess(self, X, batch_labels=None, fit=True):
        """
        Apply preprocessing pipeline.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignments (optional)
            fit: Whether to fit preprocessing components

        Returns:
            Preprocessed expression matrix
        """
        X = np.array(X, dtype=float)

        # Step 1: KNN Imputation (if missing values)
        if np.any(np.isnan(X)):
            print("Imputing missing values...")
            X = self.knn_imputer.fit_transform(X)

        # Step 2: Quantile Normalization
        print("Applying quantile normalization...")
        if fit:
            X = self.quantile_normalizer.fit_transform(X)
        else:
            X = self.quantile_normalizer.transform(X)

        # Step 3: ComBat Batch Correction (if batch labels provided)
        if batch_labels is not None:
            print("Applying ComBat batch correction...")
            if fit:
                self.combat_corrector = ComBatCorrection()
                X = self.combat_corrector.fit_transform(X, batch_labels)
            elif self.combat_corrector is not None:
                X = self.combat_corrector.transform(X, batch_labels)

        return X

    def select_features(self, X, y):
        """
        Apply attention-weighted feature selection with RFECV.

        Args:
            X: Preprocessed expression matrix
            y: Class labels

        Returns:
            Reduced expression matrix, selected gene indices
        """
        print("\nFeature Selection Pipeline")
        print("-" * 40)

        # Step 1: Compute attention scores
        print("Computing attention scores...")
        self.attention_selector.fit(X, y)

        # Step 2: Pre-select top genes based on attention
        n_preselect = min(5000, X.shape[1])
        top_genes = self.attention_selector.get_top_genes(n_preselect)
        X_preselected = X[:, top_genes]

        print(f"Pre-selected {len(top_genes)} genes based on attention scores")

        # Step 3: RFECV for final selection
        print("Running RFECV...")
        base_estimator = RandomForestClassifier(
            n_estimators=100,
            random_state=self.random_state,
            n_jobs=-1
        )

        rfecv = RFECV(
            estimator=base_estimator,
            step=self.rfecv_step,
            cv=StratifiedKFold(self.cv_folds, shuffle=True, random_state=self.random_state),
            scoring='accuracy',
            n_jobs=-1,
            min_features_to_select=min(self.n_selected_features, X_preselected.shape[1])
        )

        rfecv.fit(X_preselected, y)

        # Get selected genes
        rfecv_selected = rfecv.support_
        final_genes_in_preselected = np.where(rfecv_selected)[0]
        self.selected_genes = top_genes[final_genes_in_preselected]

        print(f"RFECV selected {len(self.selected_genes)} genes")
        print(f"Optimal CV score: {rfecv.cv_results_['mean_test_score'].max():.4f}")

        # Store training info
        self.training_info['n_genes_original'] = X.shape[1]
        self.training_info['n_genes_after_attention'] = len(top_genes)
        self.training_info['n_genes_final'] = len(self.selected_genes)
        self.training_info['feature_reduction'] = 100 * (1 - len(self.selected_genes) / X.shape[1])

        return X[:, self.selected_genes], self.selected_genes

    def fit(self, X, y, batch_labels=None, gene_names=None):
        """
        Train the complete MICRO-AI pipeline.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels
            batch_labels: Batch assignments (optional)
            gene_names: Gene identifiers (optional)
        """
        print("=" * 60)
        print("MICRO-AI Training Pipeline")
        print("=" * 60)

        X = np.array(X, dtype=float)
        self.gene_names = gene_names

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)
        self.n_classes = len(np.unique(y_encoded))
        self.classes_ = np.unique(y)



        print(f"\nDataset: {X.shape[0]} samples, {X.shape[1]} genes, {self.n_classes} classes")

        # Preprocessing
        print("\n--- Preprocessing ---")
        X_preprocessed = self.preprocess(X, batch_labels, fit=True)

        # Feature selection
        print("\n--- Feature Selection ---")
        X_selected, _ = self.select_features(X_preprocessed, y_encoded)

        # Standardization
        print("\n--- Standardization ---")
        X_scaled = self.scaler.fit_transform(X_selected)

        # Train ensemble
        print("\n--- Ensemble Training ---")
        self.ensemble = AdaptiveEnsembleClassifier(
            calibrate=self.calibrate,
            random_state=self.random_state
        )
        self.ensemble.fit(X_scaled, y_encoded)

        print("\n" + "=" * 60)
        print("Training Complete")
        print("=" * 60)

        return self

    def predict(self, X, batch_labels=None):
        """
        Predict class labels for new samples.

        Args:
            X: Expression matrix
            batch_labels: Batch assignments (optional)

        Returns:
            Predicted class labels (original encoding)
        """
        X = np.array(X, dtype=float)

        # Preprocessing
        X_preprocessed = self.preprocess(X, batch_labels, fit=False)

        # Select features
        X_selected = X_preprocessed[:, self.selected_genes]

        # Standardization
        X_scaled = self.scaler.transform(X_selected)

        # Predict
        y_pred_encoded = self.ensemble.predict(X_scaled)

        # Decode labels
        y_pred = self.label_encoder.inverse_transform(y_pred_encoded)

        return y_pred

    def predict_proba(self, X, batch_labels=None):
        """
        Predict class probabilities for new samples.

        Args:
            X: Expression matrix
            batch_labels: Batch assignments (optional)

        Returns:
            Class probabilities
        """
        X = np.array(X, dtype=float)

        # Preprocessing
        X_preprocessed = self.preprocess(X, batch_labels, fit=False)

        # Select features
        X_selected = X_preprocessed[:, self.selected_genes]

        # Standardization
        X_scaled = self.scaler.transform(X_selected)

        # Predict probabilities
        proba = self.ensemble.predict_proba(X_scaled)

        return proba

    def get_selected_gene_names(self):
        """Get names of selected genes."""
        if self.gene_names is not None and self.selected_genes is not None:
            return [self.gene_names[i] for i in self.selected_genes]
        return self.selected_genes

    def get_feature_importance(self):
        """Get feature importance for selected genes."""
        if self.ensemble is None:
            return None
        return self.ensemble.get_feature_importance()

    def get_attention_scores(self):
        """Get attention scores for all genes."""
        return self.attention_selector.attention_scores


##xAI

In [ ]:
import io
import os
import contextlib

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

In [ ]:
MODEL_PATH = "/content/GSE7390_model.joblib"
GENES_CSV = "/content/GSE7390_demo_x_real.csv"  # samples x ALL genes
OUT_DIR = "xai_output"

N_BACKGROUND = 50     # reference samples SHAP compares against
N_PER_SAMPLE = 5      # how many individual samples get their own explanation
N_GLOBAL = 30         # how many samples are used for the global gene ranking
TOP_K = 15

In [ ]:
def quiet(fn, *args, **kwargs):
    """MICROAI.preprocess prints a lot; silence it."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

In [ ]:
def to_model_space(model, X_raw):
    """
    Raw expression (samples x ALL genes)  ->  what the classifier really sees:
    preprocess -> keep selected genes -> standardise (so values are z-scores).
    Mirrors MICROAI.predict_proba, step for step.
    (batch_labels=None because the model was fitted without batch labels.)
    """
    X_pre = quiet(model.preprocess, np.asarray(X_raw, dtype=float),
                  batch_labels=None, fit=False)
    return model.scaler.transform(X_pre[:, model.selected_genes])



In [ ]:
def build_explainer(model, X_background_raw, gene_names, n_background=50, seed=0):
    """
    Model-agnostic SHAP explainer on the FULL ensemble output
    (GBM + RF + SVM + weights + isotonic calibration), for P(classes_[1]).
    """
    Z_bg = to_model_space(model, X_background_raw)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Z_bg), size=min(n_background, len(Z_bg)), replace=False)

    def f(Z):
        return model.ensemble.predict_proba(Z)[:, 1]

    masker = shap.maskers.Independent(Z_bg[idx], max_samples=len(idx))
    return shap.Explainer(f, masker, algorithm="permutation",
                          feature_names=list(gene_names), seed=seed)

In [ ]:
def explain(model, explainer, X_raw, n_features):
    Z = to_model_space(model, X_raw)
    max_evals = max(1000, 2 * n_features + 1)   # permutation SHAP needs >= 2*features+1
    return explainer(Z, max_evals=max_evals, silent=True)

In [ ]:
def sample_table(exp_row, class_labels, top_k=15):
    """Per-sample 'why' table, strongest genes first."""
    df = pd.DataFrame({
        "gene": exp_row.feature_names,
        "z_expression": exp_row.data,        # how high/low vs. the training average
        "shap": exp_row.values,              # contribution to P(classes_[1])
    })
    df["pushes_toward"] = np.where(df["shap"] > 0,
                                   f"class {class_labels[1]}",
                                   f"class {class_labels[0]}")
    order = df["shap"].abs().sort_values(ascending=False).index
    return df.loc[order].head(top_k).reset_index(drop=True)

In [ ]:
def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    # ---- load model + data ------------------------------------------------
    model = joblib.load(MODEL_PATH)
    genes = pd.read_csv(GENES_CSV, index_col=0)          # rows = samples, cols = genes/probes
    X_all = genes.to_numpy(dtype=float)

    # gene_names was never passed to model.fit(), so map indices -> names yourself
    sel_names = genes.columns.to_numpy()[model.selected_genes]
    n_feat = len(sel_names)
    classes = list(model.classes_)

    w = model.ensemble.weights
    print(f"{n_feat} selected genes | classes = {classes}")
    print(f"Ensemble weights  GBM={w[0]:.3f}  RF={w[1]:.3f}  SVM={w[2]:.3f}")
    if w[2] > 0.5:
        print("  -> SVM dominates, so tree-based importances would be misleading; "
              "SHAP below explains the actual ensemble output.")

    # ---- build explainer (background = training-like samples) --------------
    explainer = build_explainer(model, X_all, sel_names, n_background=N_BACKGROUND)

    # ---- A) local: why did the model decide this for THIS sample? ----------
    rng = np.random.default_rng(1)
    pick = rng.choice(len(X_all), size=min(N_PER_SAMPLE, len(X_all)), replace=False)
    exp_local = explain(model, explainer, X_all[pick], n_feat)

    for k, i in enumerate(pick):
        p = float(exp_local.base_values[k] + exp_local.values[k].sum())
        label = classes[1] if p >= 0.5 else classes[0]
        print(f"\n=== {genes.index[i]} -> predicted {label} "
              f"(P(class {classes[1]}) ~ {p:.3f}, baseline {exp_local.base_values[k]:.3f}) ===")
        tbl = sample_table(exp_local[k], classes, TOP_K)
        print(tbl.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
        tbl.to_csv(f"{OUT_DIR}/explain_{genes.index[i]}.csv", index=False)

        plt.figure()
        shap.plots.waterfall(exp_local[k], max_display=TOP_K, show=False)
        plt.title(f"{genes.index[i]}: which genes drove the prediction")
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/waterfall_{genes.index[i]}.png", dpi=150, bbox_inches="tight")
        plt.close()

    # ---- B) global: which genes matter across samples? ---------------------
    g_idx = rng.choice(len(X_all), size=min(N_GLOBAL, len(X_all)), replace=False)
    exp_global = explain(model, explainer, X_all[g_idx], n_feat)

    ranking = (pd.Series(np.abs(exp_global.values).mean(0), index=sel_names,
                         name="mean_abs_shap")
               .sort_values(ascending=False))
    ranking.to_csv(f"{OUT_DIR}/global_gene_ranking.csv")
    print("\nTop genes overall (mean |SHAP|):")
    print(ranking.head(TOP_K).to_string())

    plt.figure()
    shap.plots.bar(exp_global, max_display=20, show=False)
    plt.savefig(f"{OUT_DIR}/global_bar.png", dpi=150, bbox_inches="tight")
    plt.close()

    plt.figure()
    shap.plots.beeswarm(exp_global, max_display=20, show=False)
    plt.savefig(f"{OUT_DIR}/global_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()

    print(f"\nSaved tables and plots to ./{OUT_DIR}/")


if __name__ == "__main__":
    main()

250 selected genes | classes = [np.int64(0), np.int64(1)]
Ensemble weights  GBM=0.000  RF=0.000  SVM=1.000
  -> SVM dominates, so tree-based importances would be misleading; SHAP below explains the actual ensemble output.

=== GSM177950 -> predicted 0 (P(class 1) ~ 0.059, baseline 0.092) ===
       gene  z_expression   shap pushes_toward
  219225_at        -1.378 -0.030       class 0
206457_s_at        -0.169 +0.016       class 1
  210516_at        -1.874 +0.014       class 1
  216478_at        -0.859 -0.012       class 0
  204818_at        -1.519 -0.012       class 0
204409_s_at        -0.777 -0.011       class 0
211348_s_at        -1.904 -0.011       class 0
  206952_at        -2.901 +0.010       class 1
  203685_at        +1.190 +0.010       class 1
  219084_at        -0.612 -0.009       class 0
  219830_at        -0.526 +0.009       class 1
203881_s_at        -0.806 -0.009       class 0
204745_x_at        +0.598 -0.008       class 0
  207607_at        -3.444 +0.008       class 1
  